# Track C — Faster R-CNN ResNet-50 FPN

Notebook tái lập và phân tích run canonical trên subset 3.000 ảnh của Track A. Runner chính là `track_c_faster_rcnn.py`; checkpoint atomic và resume sau mỗi epoch.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
CONFIG = ROOT / 'track_c_config_track_a.json'
ARTIFACT = ROOT / 'artifacts' / 'track_c' / 'track_a_subset'
config = json.loads(CONFIG.read_text(encoding='utf-8'))
config

## Dữ liệu và giao thức

Nguồn là Roboflow project `vietnam-traffic-sign-detection-2i2j8`, version 5. Run canonical dùng 3.000 ảnh train Track A, giữ nguyên 784 validation và 613 test. `data.yaml` khai báo 58 lớp; train nguồn chỉ có mẫu cho 56 lớp.

In [ ]:
audit = json.loads((ARTIFACT / 'dataset_audit.json').read_text(encoding='utf-8'))
pd.DataFrame(audit['splits']).T[['images', 'label_files', 'instances', 'empty_or_missing_labels']]

## Train/resume và đánh giá

Để chạy lại: `python -u track_c_faster_rcnn.py all --config track_c_config_track_a.json`. Nếu checkpoint tồn tại, runner resume từ epoch kế tiếp. Cấu hình: COCO pretrained, SGD lr=0.005, momentum=0.9, weight decay=0.0005, seed=42, batch 4 FP32, 18 epoch.

In [ ]:
history = pd.read_csv(ARTIFACT / 'train_history.csv')
ax = history.plot(x='epoch', y='loss', marker='o', grid=True, title='Faster R-CNN training loss')
ax.set_ylabel('mean total loss')
plt.show()
history.tail()

In [ ]:
metrics = json.loads((ARTIFACT / 'fasterrcnn_metrics.json').read_text(encoding='utf-8'))
pd.Series({k: metrics[k] for k in ['map50', 'map50_95', 'map_small', 'map_medium', 'map_large', 'fps', 'num_params', 'train_time_min']})

## So sánh cùng RTX 4060

YOLO và Faster R-CNN được đánh giá lại trên cùng validation và 100 ảnh test. Checkpoint YOLO baseline 640 dùng đúng subset Track A 3.000 ảnh; bảng chính thức là phép so sánh kiểm soát dữ liệu và phần cứng RTX 4060. Hai model vẫn khác số epoch và transform đầu vào, cần ghi rõ khi diễn giải.

In [ ]:
pd.read_csv(ROOT / 'artifacts' / 'track_c' / 'comparison_rtx4060.csv')

## Kiểm chứng và ảnh inference

`checkpoint_verification.json` chứa kết quả nạp lại checkpoint trong tiến trình mới và SHA-256. Ảnh minh họa nằm trong `data/BTL_DeTai4_9000/runs_rcnn/track_a_subset/inference_examples/`.

In [ ]:
verification = json.loads((ROOT / 'artifacts' / 'track_c' / 'checkpoint_verification.json').read_text(encoding='utf-8'))
verification['track_a_subset']